In [1]:
!pip install numpy==1.26.4 --force-reinstall --no-cache-dir
!pip install pandas==2.1.4 pyarrow==14.0.2 --force-reinstall --no-cache-dir
!pip install scikit-learn==1.3.2 --force-reinstall --no-cache-dir

!pip install datasets==2.19.2 --force-reinstall --no-cache-dir
!pip install huggingface-hub==0.20.3 --force-reinstall --no-cache-dir
!pip install transformers==4.38.2 accelerate==0.30.1 --force-reinstall --no-cache-dir


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 324.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.4.1 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires notebook==6.5.7, but you have notebook 6.5.4 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.3, but

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/intent-data/autotrain_complaints.csv


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch


ValueError: All ufuncs must have type `numpy.ufunc`. Received (<ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>)

In [ ]:
df = pd.read_csv("/kaggle/input/intent-data/autotrain_complaints.csv")

df.head()


In [ ]:
sorted(df["label"].unique())

In [ ]:
def map_category(label):
    text = label.lower()

    # -------------------------
    # 1. ACCOUNT MANAGEMENT
    # -------------------------
    if any(k in text for k in [
        "account", "opening", "closing", "managing", "profile", "login", "password"
    ]):
        return "Account Management"

    # -------------------------
    # 2. BILLING ISSUES
    # -------------------------
    if any(k in text for k in [
        "fee", "interest", "late", "charge", "billing", "statement"
    ]):
        return "Billing Issues"

    # -------------------------
    # 3. TRANSACTION ISSUES
    # -------------------------
    if any(k in text for k in [
        "transaction", "money was", "wrong amount", "payment not", "refund", "transfer"
    ]):
        return "Transaction Issues"

    # -------------------------
    # 4. PAYMENT PROBLEMS
    # -------------------------
    if any(k in text for k in [
        "struggling to pay", "can't repay", "unable to pay", "payment", "auto debit"
    ]):
        return "Payment Problems"

    # -------------------------
    # 5. CARD ISSUES
    # -------------------------
    if any(k in text for k in [
        "card", "atm", "debit card", "credit card", "trouble using"
    ]):
        return "Card Issues"

    # -------------------------
    # 6. LOAN ISSUES
    # -------------------------
    if any(k in text for k in [
        "loan", "mortgage", "foreclosure", "escrow", "lease"
    ]):
        return "Loan Issues"

    # -------------------------
    # 7. CREDIT REPORT ISSUES
    # -------------------------
    if any(k in text for k in [
        "credit report", "credit score", "dispute", "fraud alert", "identity theft"
    ]):
        return "Credit Report Issues"

    # -------------------------
    # 8. CUSTOMER SERVICE
    # -------------------------
    if any(k in text for k in [
        "customer service", "customer relations", "communication", "support"
    ]):
        return "Customer Service"

    # -------------------------
    # 9. DEBT COLLECTION
    # -------------------------
    if any(k in text for k in [
        "debt", "collection", "threat", "harass"
    ]):
        return "Debt Collection Issues"

    # -------------------------
    # 10. FRAUD / SCAM
    # -------------------------
    if any(k in text for k in [
        "fraud", "scam", "unauthorized", "identity theft"
    ]):
        return "Fraud / Scam"

    # -------------------------
    # 11. ADVERTISING / MISLEADING
    # -------------------------
    if any(k in text for k in [
        "advertising", "marketing", "misleading", "promotion"
    ]):
        return "Advertising & Misleading Info"

    # -------------------------
    # 12. OTHER ISSUES (fallback)
    # -------------------------
    return "Other Issues"

# APPLY MAPPING
df["intent_category"] = df["label"].apply(map_category)


In [ ]:
df["intent_category"].value_counts()


In [ ]:
import pandas as pd

def balance_dataset(df, max_per_class=5000):
    df_balanced = (
        df.groupby("intent_category")
          .apply(lambda x: x.sample(max_per_class, replace=False) 
                if len(x) > max_per_class else x)
          .reset_index(drop=True)
    )
    return df_balanced

df_balanced = balance_dataset(df, max_per_class=5000)
df_balanced.intent_category.value_counts()


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


In [ ]:
df_balanced.head()

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

df = df_balanced.copy()

# Encode labels
labels = sorted(df["intent_category"].unique())

label2id = {v: i for i, v in enumerate(labels)}
id2label = {i: v for v, i in label2id.items()}

df["label_id"] = df["intent_category"].map(label2id)


# Train-test split
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["label_id"],
    random_state=42
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]]).rename_column("label_id", "labels")
val_dataset   = Dataset.from_pandas(val_df[["text", "label_id"]]).rename_column("label_id", "labels")

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch")
val_dataset.set_format("torch")


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# FORCE GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model running on:", device)


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="/kaggle/working/intent_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    fp16=True,             # 🔥 enables GPU half precision
    optim="adamw_torch",   # 🔥 faster optimizer
    logging_steps=100,
    report_to="none"
)


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("🚀 Training started using GPU...")
trainer.train()

model.save_pretrained("/kaggle/working/intent_model")
tokenizer.save_pretrained("/kaggle/working/intent_model")

print("✔ Model saved!")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

MODEL_PATH = "/kaggle/working/intent_model"

# Load tokenizer + model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

model.eval()

if torch.cuda.is_available():
    model.to("cuda")
    print("🔥 Model loaded on GPU")
else:
    print("⚠ Running on CPU")


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-classification",
    model="/kaggle/working/intent_model",
    tokenizer="/kaggle/working/intent_model",
    device=0  # forces GPU
)

tests = [
    "I want to check my account balance",
    "My credit card was charged wrongly!",
    "Please close my loan account",
    "How can I change my password?",
    "The service is terrible, I want to complain.",
    "Where is my refund?"
]

for t in tests:
    print(pipe(t)[0])


In [ ]:
!rm -rf /kaggle/working/intent_model


In [ ]:
!ls /kaggle/working


In [ ]:
!nvidia-smi
